In [ ]:
# If you want to run everything automatically, you can run this ipynb

import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

###############################################################
###############################################################
###############################################################

# Choices
stages = [
    '4_instruction_register', # Parallel
    '6_test_case',
    '7_repair_test_case', # Optional, sequential
    '8_emulator', # Parallel
]

START_FROM = '4_instruction_register'
SKIP_REPAIR = False
MAX_ATTEMPTS = 10

def run_notebook_once(notebook):
    path = Path(notebook).resolve()

    print(f'\n===== RUNNING {notebook} =====\n')

    result = subprocess.run([
        'jupyter',
        'nbconvert',
        '--to', 'notebook',
        '--execute',
        '--inplace',
        '--ExecutePreprocessor.timeout=-1',
        path.name,
    ], cwd=path.parent)

    if result.returncode != 0:
        print(f'\nFAILED at {notebook}')
        return False

    print(f'\nFINISHED {notebook}')
    return True

def run_notebook_retry(notebook):
    for attempt in range(1, MAX_ATTEMPTS + 1):
        print(
            f'\n===== ATTEMPT '
            f'{attempt}/{MAX_ATTEMPTS}: '
            f'{notebook} =====\n'
        )

        if run_notebook_once(notebook):
            return True

    return False

def run_parallel(notebooks):
    failed = []

    with ThreadPoolExecutor(
        max_workers=len(notebooks)
    ) as executor:
        futures = {
            executor.submit(
                run_notebook_retry,
                notebook,
            ): notebook
            for notebook in notebooks
        }

        for future in as_completed(futures):
            notebook = futures[future]

            if not future.result():
                failed.append(notebook)

    if not failed:
        return True

    print(
        '\nParallel retries exhausted for: '
        + ', '.join(failed)
    )

    print(
        '\nRetrying failed notebooks '
        'sequentially...\n'
    )

    for notebook in failed:
        if not run_notebook_once(notebook):
            print(
                f'\nFINAL FAILURE at {notebook}'
            )
            return False

    return True

def run_sequential(notebooks):
    for notebook in notebooks:
        if not run_notebook_retry(notebook):
            return False

    return True

if START_FROM not in stages:
    raise ValueError(
        f'Invalid START_FROM: {START_FROM}'
    )

start_index = stages.index(START_FROM)

if start_index <= stages.index('4_instruction_register'):
    if not run_parallel([
        '4_INSTRUCTION.ipynb',
        '5_REGISTER.ipynb',
    ]):
        raise SystemExit(1)

if start_index <= stages.index('6_test_case'):
    if not run_notebook_retry(
        '6_TEST_CASE.ipynb'
    ):
        raise SystemExit(1)

if start_index <= stages.index('7_repair_test_case'):
    if SKIP_REPAIR:
        print('\n===== SKIPPING REPAIR TEST CASE =====\n')
    else:
        if not run_sequential([
            '7_REPAIR_TEST_CASE/REPAIR_TEST_CASE.ipynb',
            '7_REPAIR_TEST_CASE_2/REPAIR_TEST_CASE_2.ipynb',
            '7_REPAIR_TEST_CASE_3/REPAIR_TEST_CASE_3.ipynb',
        ]):
            raise SystemExit(1)

if start_index <= stages.index('8_emulator'):
    if not run_parallel([
        '8_EMULATOR/2_QEMU/QEMU.ipynb',
        '8_EMULATOR/3_SPIKE/SPIKE.ipynb',
        '8_EMULATOR/4_RVVM/RVVM.ipynb',
        '8_EMULATOR/5_RENODE/RENODE.ipynb',
    ]):
        raise SystemExit(1)

print('\n====')
print('DONE')
print('====')